# Read the Floor — Colab pipeline (Skalski / SAM2)

Faithful port of Roboflow's open-source basketball notebook
*(detect, track & identify players)* + the make/miss shot detector. It extracts player
coordinates from a clip; **our brain** then reads the floor (who guards whom + Beaten Index)
and renders the X-ray overlay.

```
video -> RF-DETR (detect) -> SAM2 real-time (track) -> SigLIP/UMAP/K-means (teams)
      -> SmolVLM2 jersey OCR -> court keypoints -> homography (-> FEET) -> frames_out
      -> Skalski footage overlay (masks + boxes + jersey numbers on the clip)
      -> [skalski adapter] -> tracking table -> brain -> render (+ ShotEventTracker make/miss)
```

**Runtime -> Change runtime type -> GPU (A100 ideal; T4 only with SAM2 `tiny` + OCR off).**
Run cells top to bottom.

**Before you run:** push the latest `read_the_floor/` to the GitHub repo's `main` branch
(cell 3 clones from GitHub, not your local disk). Secrets (key icon): add `ROBOFLOW_API_KEY`.


## 1. Install SAM2 real-time + download checkpoints

Uses `Gy920/segment-anything-2-real-time` (the exact fork Skalski uses). The editable-wheel build of the optional `sam2._C` CUDA extension may print a red ERROR — that is harmless; the `build_ext --inplace` line right after rebuilds it correctly.

In [ ]:
!git clone https://github.com/Gy920/segment-anything-2-real-time.git
%cd /content/segment-anything-2-real-time
!pip install -e . -q
!python setup.py build_ext --inplace
!(cd checkpoints && bash download_ckpts.sh)
%cd /content


## 2. Install the rest of the stack

`inference-gpu` runs the Roboflow models locally on the GPU (no pycuda/hosted hop). `sports@feat/basketball` ships `CourtConfiguration(NBA, FEET)` — 33 court vertices already in feet, matching the keypoint model. The `--force-reinstall pillow` line avoids Colab's half-upgraded-Pillow `_Ink` ImportError.

In [ ]:
!pip install -q gdown inference-gpu
!pip install -q supervision==0.27.0
!pip install -q git+https://github.com/roboflow/sports.git@feat/basketball
!pip install -q transformers num2words imageio matplotlib
# Colab sometimes ends up with a half-upgraded Pillow (ImportError: cannot import name '_Ink').
# Force one coherent build so `import supervision` works without a runtime restart.
!pip install -q --upgrade "pillow>=12.1.0"   # Pillow 12.0.0 is broken (ImportError: _Ink); 12.1.0+ fixes it

import os
# force ONNX Runtime onto CUDA so get_model() runs on the GPU (Skalski cell)
os.environ["ONNXRUNTIME_EXECUTION_PROVIDERS"] = "[CUDAExecutionProvider]"
print("deps installed")


## 3. Clone our repo + import the brain/adapter

Our modules are pure-Python (schema, brain, render, adapter). Added to `sys.path` by absolute path so they import regardless of the current directory.

In [ ]:
!git clone https://github.com/aldemirkonuk/TalentBetweenTheLines.git
import sys
RTF = "/content/TalentBetweenTheLines/read_the_floor"
sys.path.insert(0, RTF)

# If the import below fails, push your latest read_the_floor/ to the repo's main branch
# first — this notebook clones from GitHub, not from your local disk.
import schema, brain, render, basketball_court_config
from adapters import skalski

print("our modules OK")
# NOTE: the live keypoint model emits 33 points; our basketball_court_config.py is 16
# (our own reference court). For the homography we use the sports 33-pt FEET config
# below — same units (0..94 x 0..50), correct index order for THIS model.
print("our reference court vertices:", basketball_court_config.vertices_ft().shape)


## 4. Secrets, GPU check, and the clip

In [ ]:
import os
from google.colab import userdata

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
print("key set:", bool(os.environ.get("ROBOFLOW_API_KEY")))

import torch
print("CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

VIDEO = "/content/drive/MyDrive/Colab Notebooks/clip.mp4"   # your possession clip
assert os.path.exists(VIDEO), f"Not found: {VIDEO}"
print("clip OK —", round(os.path.getsize(VIDEO) / 1e6, 1), "MB")


## 5. Load models + court config

RF-DETR detector, court-keypoint model, jersey-OCR model, team-classifier deps, and the **feet** court config. Player-model class ids: `ball=0`, `ball-in-basket=1`, `number=2`, players=`[3,4,5,6,7]` (Skalski's exact ids/thresholds).

In [ ]:
import numpy as np
import supervision as sv
from inference import get_model
from tqdm import tqdm
from typing import List, Tuple

from sports import (
    TeamClassifier, ViewTransformer, MeasurementUnit, ConsecutiveValueTracker, clean_paths)
from sports.basketball import (
    CourtConfiguration, League, draw_court, draw_points_on_court, draw_paths_on_court,
    ShotEventTracker)

# --- detector (RF-DETR) — Skalski thresholds ---
PLAYER_DETECTION_MODEL_ID = "basketball-player-detection-3-ycjdo/4"
PLAYER_DETECTION_MODEL_CONFIDENCE = 0.4
PLAYER_DETECTION_MODEL_IOU_THRESHOLD = 0.9
PLAYER_DETECTION_MODEL = get_model(model_id=PLAYER_DETECTION_MODEL_ID)

# --- court keypoints — Skalski thresholds ---
KEYPOINT_DETECTION_MODEL_ID = "basketball-court-detection-2/14"
KEYPOINT_DETECTION_MODEL_CONFIDENCE = 0.3
KEYPOINT_DETECTION_MODEL_ANCHOR_CONFIDENCE = 0.5
KEYPOINT_DETECTION_MODEL = get_model(model_id=KEYPOINT_DETECTION_MODEL_ID)

# --- jersey numbers (SmolVLM2 OCR) ---
NUMBER_RECOGNITION_MODEL_ID = "basketball-jersey-numbers-ocr/3"
NUMBER_RECOGNITION_MODEL = get_model(model_id=NUMBER_RECOGNITION_MODEL_ID)
NUMBER_RECOGNITION_MODEL_PROMPT = "Read the number."

# --- class ids in the player model ---
BALL_CLASS_ID = 0          # our addition (Skalski's identify notebook maps players only)
BALL_IN_BASKET_CLASS_ID = 1
NUMBER_CLASS_ID = 2
PLAYER_CLASS_IDS = [3, 4, 5, 6, 7]   # player + in-possession/jump/layup/block
JUMP_SHOT_CLASS_ID = 5
LAYUP_DUNK_CLASS_ID = 6
BALL_CONF = 0.2            # ball is small/fast -> lower threshold (our addition)

# --- court target in FEET (0..94 x 0..50), 33 vertices matching the model ---
config = CourtConfiguration(league=League.NBA, measurement_unit=MeasurementUnit.FEET)
print("court vertices (feet):", np.array(config.vertices).shape)


## 6. Build the SAM2 tracker (Skalski-verbatim)

`large` is faithful to Skalski (best masks; needs ~A100). **On a T4, switch to `tiny`** (commented) or you will OOM. The import preamble drops any PyPI `sam2` that `inference-gpu` may have already loaded and forces the Gy920 real-time fork (the only one with `build_sam2_camera_predictor`).

In [ ]:
%cd /content/segment-anything-2-real-time
import sys
# inference-gpu may have already imported the PyPI `sam2` (no camera predictor).
# Drop the cached module and force the Gy920 real-time fork in this directory.
sys.path.insert(0, "/content/segment-anything-2-real-time")
for _m in [m for m in list(sys.modules) if m == "sam2" or m.startswith("sam2.")]:
    del sys.modules[_m]
from sam2.build_sam import build_sam2_camera_predictor

SAM2_CHECKPOINT = "checkpoints/sam2.1_hiera_large.pt"
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"
# T4 / low-VRAM alternative:
# SAM2_CHECKPOINT = "checkpoints/sam2.1_hiera_tiny.pt"
# SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_t.yaml"

predictor = build_sam2_camera_predictor(SAM2_CONFIG, SAM2_CHECKPOINT)
%cd /content
print("SAM2 predictor ready")


In [ ]:
from __future__ import annotations


class SAM2Tracker:
    """Skalski's SAM2 real-time tracker (verbatim): prompt frame 0, propagate masks + ids."""

    def __init__(self, predictor) -> None:
        self.predictor = predictor
        self._prompted = False

    def prompt_first_frame(self, frame: np.ndarray, detections: sv.Detections) -> None:
        if len(detections) == 0:
            raise ValueError("detections must contain at least one box")

        if detections.tracker_id is None:
            detections.tracker_id = list(range(1, len(detections) + 1))

        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
            self.predictor.load_first_frame(frame)
            for xyxy, obj_id in zip(detections.xyxy, detections.tracker_id):
                bbox = np.asarray([xyxy], dtype=np.float32)
                self.predictor.add_new_prompt(
                    frame_idx=0,
                    obj_id=int(obj_id),
                    bbox=bbox,
                )

        self._prompted = True

    def propagate(self, frame: np.ndarray) -> sv.Detections:
        if not self._prompted:
            raise RuntimeError("Call prompt_first_frame before propagate")

        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
            tracker_ids, mask_logits = self.predictor.track(frame)

        tracker_ids = np.asarray(tracker_ids, dtype=np.int32)
        masks = (mask_logits > 0.0).cpu().numpy()
        masks = np.squeeze(masks).astype(bool)

        if masks.ndim == 2:
            masks = masks[None, ...]

        masks = np.array([
            sv.filter_segments_by_distance(mask, relative_distance=0.03, mode="edge")
            for mask in masks
        ])

        xyxy = sv.mask_to_xyxy(masks=masks)
        detections = sv.Detections(xyxy=xyxy, mask=masks, tracker_id=tracker_ids)
        return detections

    def reset(self) -> None:
        self._prompted = False


print("SAM2Tracker defined")


## 7. Fit the team classifier (SigLIP -> UMAP -> K-means)

Skalski's recipe: stride-sample crops, scale boxes 0.4, fit once, freeze. Team ids (0/1) are arbitrary; the brain only needs two stable groups.

In [ ]:
STRIDE = 30
crops = []
for f in tqdm(sv.get_video_frames_generator(VIDEO, stride=STRIDE), desc="team crops"):
    result = PLAYER_DETECTION_MODEL.infer(
        f, confidence=PLAYER_DETECTION_MODEL_CONFIDENCE,
        iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD, class_agnostic_nms=True)[0]
    detections = sv.Detections.from_inference(result)
    detections = detections[np.isin(detections.class_id, PLAYER_CLASS_IDS)]
    for box in sv.scale_boxes(xyxy=detections.xyxy, factor=0.4):
        crops.append(sv.crop_image(f, box))

assert len(crops) >= 2, "not enough player crops to fit teams — check the clip / detections"
team_classifier = TeamClassifier(device="cuda")
team_classifier.fit(crops)
print("team classifier fit on", len(crops), "crops")


## 8. (Diagnostic) court keypoints on one frame

Sanity check: how many of the 33 court points the model sees in your clip. Need >= 4 above the anchor threshold for a homography.

In [ ]:
frame = next(sv.get_video_frames_generator(VIDEO, stride=30))
result = KEYPOINT_DETECTION_MODEL.infer(frame, confidence=KEYPOINT_DETECTION_MODEL_CONFIDENCE)[0]
key_points = sv.KeyPoints.from_inference(result)
mask = key_points.confidence[0] > KEYPOINT_DETECTION_MODEL_ANCHOR_CONFIDENCE
print("keypoints above", KEYPOINT_DETECTION_MODEL_ANCHOR_CONFIDENCE, ":",
      int(np.count_nonzero(mask)), "/ 33")


## 9. Run the pipeline -> `frames_out`

Skalski's detect/track/identify + map loop, assembled into our per-frame contract
`(players, ball, transformer, team_by_id, number_by_id)`:

- **players**: SAM2 tracks (stable ids prompted from frame 0) — Skalski cells 21/46
- **transformer**: per-frame homography to **feet** — Skalski cell 46 (verbatim), reuses last good fit if sparse
- **team_by_id**: `TEAMS` from the first SAM2 prompt frame, keyed by tracker_id — Skalski cell 46
- **number_by_id**: SmolVLM2 jersey OCR, mask-IoS match + `ConsecutiveValueTracker(3)` vote — Skalski cells 34/36/39
- **ball**: per-frame RF-DETR at low confidence (`class_id == 0`) — *our addition* for possession

Toggles: `RUN_OCR=False` to skip OCR on low-VRAM GPUs; `MAX_FRAMES=60` for a quick smoke test.


In [ ]:
RUN_OCR = True          # set False on a T4 if SmolVLM2 + SAM2 OOM
OCR_EVERY = 5           # Skalski cadence (index % 5 == 0)
MAX_FRAMES = None       # set e.g. 60 for a quick smoke test; None = whole clip


def coords_above_threshold(matrix, threshold, sort_desc=True):
    """Skalski cell 34 (verbatim): (row, col) where IoS > threshold, sorted desc."""
    A = np.asarray(matrix)
    rows, cols = np.where(A > threshold)
    pairs = list(zip(rows.tolist(), cols.tolist()))
    if sort_desc:
        pairs.sort(key=lambda rc: A[rc[0], rc[1]], reverse=True)
    return pairs


def fit_homography(frame):
    """Skalski cell 46 (verbatim): keypoints -> ViewTransformer to feet."""
    result = KEYPOINT_DETECTION_MODEL.infer(frame, confidence=KEYPOINT_DETECTION_MODEL_CONFIDENCE)[0]
    key_points = sv.KeyPoints.from_inference(result)
    landmarks_mask = key_points.confidence[0] > KEYPOINT_DETECTION_MODEL_ANCHOR_CONFIDENCE
    if np.count_nonzero(landmarks_mask) >= 4:
        court_landmarks = np.array(config.vertices)[landmarks_mask]
        frame_landmarks = key_points[:, landmarks_mask].xy[0]
        return ViewTransformer(source=frame_landmarks, target=court_landmarks)
    return None


def get_ball(frame):
    """Our addition: low-confidence ball detection for the brain's possession logic."""
    result = PLAYER_DETECTION_MODEL.infer(frame, confidence=BALL_CONF,
                                          iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD)[0]
    d = sv.Detections.from_inference(result)
    return d[d.class_id == BALL_CLASS_ID]


def _read_number(crop):
    """Jersey read via the high-level .infer() (preprocess -> generate -> postprocess).
    inference_models 0.29: .predict() is low-level (wants a processed dict); .infer()
    accepts a BGR numpy crop and applies the fine-tune's own pre_process_generation."""
    out = NUMBER_RECOGNITION_MODEL.infer(crop, prompt=NUMBER_RECOGNITION_MODEL_PROMPT)
    if isinstance(out, (list, tuple)):
        out = out[0] if out else ""
    # inference_models returns an InferenceResponse object; the text is in .response
    out = getattr(out, "response", out)
    return str(out).strip()


def ocr_update(frame, player_detections, validator):
    """Skalski cells 36/39: detect numbers, match to tracks by mask-IoS, feed the voter."""
    if player_detections.mask is None or len(player_detections) == 0:
        return
    frame_h, frame_w, *_ = frame.shape
    result = PLAYER_DETECTION_MODEL.infer(
        frame, confidence=PLAYER_DETECTION_MODEL_CONFIDENCE,
        iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD)[0]
    number_detections = sv.Detections.from_inference(result)
    number_detections = number_detections[number_detections.class_id == NUMBER_CLASS_ID]
    if len(number_detections) == 0:
        return
    number_detections.mask = sv.xyxy_to_mask(
        boxes=number_detections.xyxy, resolution_wh=(frame_w, frame_h))
    number_crops = [
        sv.crop_image(frame, xyxy)
        for xyxy in sv.clip_boxes(
            sv.pad_boxes(xyxy=number_detections.xyxy, px=10, py=10), (frame_w, frame_h))
    ]
    numbers = [_read_number(number_crop) for number_crop in number_crops]
    iou = sv.mask_iou_batch(
        masks_true=player_detections.mask, masks_detection=number_detections.mask,
        overlap_metric=sv.OverlapMetric.IOS)
    pairs = coords_above_threshold(iou, 0.9)
    if not pairs:
        return
    player_idx, number_idx = zip(*pairs)
    tids = [int(player_detections.tracker_id[int(i)]) for i in player_idx]
    reads = [numbers[int(j)] for j in number_idx]
    validator.update(tracker_ids=tids, values=reads)


# ---- frame 0: detect players, assign ids, classify teams (frozen), prompt SAM2 (Skalski cell 46) ----
frame_generator = sv.get_video_frames_generator(VIDEO)
frame = next(frame_generator)

result = PLAYER_DETECTION_MODEL.infer(
    frame, confidence=PLAYER_DETECTION_MODEL_CONFIDENCE,
    iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD)[0]
detections = sv.Detections.from_inference(result)
detections = detections[np.isin(detections.class_id, PLAYER_CLASS_IDS)]
assert len(detections) > 0, "no players on frame 0 — pick a clip whose first frame shows players"
detections.tracker_id = np.arange(1, len(detections.class_id) + 1)

boxes = sv.scale_boxes(xyxy=detections.xyxy, factor=0.4)
crops = [sv.crop_image(frame, box) for box in boxes]
TEAMS = np.array(team_classifier.predict(crops))
team_by_id = {int(t): int(TEAMS[i]) for i, t in enumerate(detections.tracker_id)}

tracker = SAM2Tracker(predictor)
tracker.prompt_first_frame(frame, detections)

number_validator = ConsecutiveValueTracker(n_consecutive=3)
number_by_id = {}        # shared dict; populated after the loop, seen by every frame tuple

# ---- accumulate frames_out ----
frames_out = []
prev_transformer = fit_homography(frame)
if prev_transformer is not None:
    frames_out.append((detections, get_ball(frame), prev_transformer, team_by_id, number_by_id))

for index, frame in enumerate(tqdm(frame_generator, desc="tracking"), start=1):
    if MAX_FRAMES is not None and index >= MAX_FRAMES:
        break
    player_detections = tracker.propagate(frame)
    transformer = fit_homography(frame)
    if transformer is None:
        transformer = prev_transformer
    else:
        prev_transformer = transformer
    if transformer is None:
        continue
    if RUN_OCR and index % OCR_EVERY == 0:
        try:
            ocr_update(frame, player_detections, number_validator)
        except Exception as _e:
            print("OCR disabled (jersey numbers off):", _e); RUN_OCR = False
    frames_out.append((player_detections, get_ball(frame), transformer, team_by_id, number_by_id))

# ---- finalize voted jersey numbers (identity is frame-invariant) ----
if RUN_OCR:
    ids = sorted(team_by_id.keys())
    voted = number_validator.get_validated(tracker_ids=ids)
    number_by_id.update({int(t): str(v) for t, v in zip(ids, voted) if v})

assert len(frames_out) > 0, "no frames mapped — homography never fit (need >= 4 court keypoints)"
print("frames_out:", len(frames_out), "frames | jersey numbers:", number_by_id)


## 9d. Skalski footage overlay — masks + boxes + jersey numbers (his cell 67)

This is the annotated **video on the actual footage**: per-player SAM2 masks + boxes, colored
by track, with validated jersey numbers burned on. Verbatim Skalski cell 67 (adapted only for
our `_read_number` OCR call). It runs a **fresh SAM2 pass** (re-prompts frame 0), so it costs
roughly the same time as cell 9. Set `OVERLAY_MAX_FRAMES` low for a quick preview.

In [ ]:
from IPython.display import Video

OVERLAY_MAX_FRAMES = MAX_FRAMES   # reuse smoke cap; set None for the whole clip (~2nd SAM2 pass)

COLOR = sv.ColorPalette.from_hex([
    '#ffff00', '#ff9b00', '#ff66ff', '#3399ff', '#ff66b2', '#ff8080',
    '#b266ff', '#9999ff', '#66ffff', '#33ff99', '#66ff66', '#99ff00'])
mask_annotator = sv.MaskAnnotator(color=COLOR, color_lookup=sv.ColorLookup.TRACK, opacity=0.7)
box_annotator = sv.BoxAnnotator(color=COLOR, color_lookup=sv.ColorLookup.TRACK, thickness=2)
label_annotator = sv.LabelAnnotator(color=COLOR, color_lookup=sv.ColorLookup.TRACK,
                                    text_color=sv.Color.BLACK, text_scale=0.8)
overlay_validator = ConsecutiveValueTracker(n_consecutive=3)

# fresh SAM2 pass: re-prompt frame 0 (resets the predictor), then propagate
gen = sv.get_video_frames_generator(VIDEO)
frame0 = next(gen)
_r = PLAYER_DETECTION_MODEL.infer(frame0, confidence=PLAYER_DETECTION_MODEL_CONFIDENCE,
                                  iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD)[0]
det0 = sv.Detections.from_inference(_r)
det0 = det0[np.isin(det0.class_id, PLAYER_CLASS_IDS)]
det0.tracker_id = np.arange(1, len(det0.class_id) + 1)
overlay_tracker = SAM2Tracker(predictor)
overlay_tracker.prompt_first_frame(frame0, det0)


def _annotate(frame, index):
    player_detections = overlay_tracker.propagate(frame)
    if index % 5 == 0:
        frame_h, frame_w, *_ = frame.shape
        _r = PLAYER_DETECTION_MODEL.infer(frame, confidence=PLAYER_DETECTION_MODEL_CONFIDENCE,
                                          iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD)[0]
        nd = sv.Detections.from_inference(_r)
        nd = nd[nd.class_id == NUMBER_CLASS_ID]
        if len(nd):
            nd.mask = sv.xyxy_to_mask(boxes=nd.xyxy, resolution_wh=(frame_w, frame_h))
            crops = [sv.crop_image(frame, xyxy) for xyxy in
                     sv.clip_boxes(sv.pad_boxes(xyxy=nd.xyxy, px=10, py=10), (frame_w, frame_h))]
            numbers = [_read_number(c) for c in crops]
            iou = sv.mask_iou_batch(masks_true=player_detections.mask, masks_detection=nd.mask,
                                    overlap_metric=sv.OverlapMetric.IOS)
            pairs = coords_above_threshold(iou, 0.9)
            if pairs:
                pi, ni = zip(*pairs)
                tids = [int(player_detections.tracker_id[int(i)]) for i in pi]
                reads = [numbers[int(j)] for j in ni]
                overlay_validator.update(tracker_ids=tids, values=reads)
    out = frame.copy()
    out = mask_annotator.annotate(scene=out, detections=player_detections)
    out = box_annotator.annotate(scene=out, detections=player_detections)
    labels = overlay_validator.get_validated(tracker_ids=player_detections.tracker_id)
    labels = [str(l) if l else '' for l in labels]
    out = label_annotator.annotate(scene=out, detections=player_detections, labels=labels)
    return out


video_info = sv.VideoInfo.from_video_path(VIDEO)
OVERLAY_OUT = '/content/skalski_footage.mp4'
with sv.VideoSink(OVERLAY_OUT, video_info) as sink:
    sink.write_frame(_annotate(frame0, 0))
    for index, frame in enumerate(tqdm(gen, desc='overlay'), start=1):
        if OVERLAY_MAX_FRAMES is not None and index >= OVERLAY_MAX_FRAMES:
            break
        sink.write_frame(_annotate(frame, index))

OVERLAY_OUT_C = '/content/skalski_footage_compressed.mp4'
!ffmpeg -y -loglevel error -i {OVERLAY_OUT} -vcodec libx264 -crf 28 {OVERLAY_OUT_C}
print('saved', OVERLAY_OUT_C)
Video(OVERLAY_OUT_C, embed=True)


## 10. Adapter -> tracking table

`skalski.frame_to_rows` projects each ground-contact point to court feet; `build_table` smooths per track. Then we validate the schema and run the plan's inline asserts.

In [ ]:
per_frame = [
    skalski.frame_to_rows(i, players, ball, transformer, team_by_id, number_by_id)
    for i, (players, ball, transformer, team_by_id, number_by_id) in enumerate(frames_out)
]
df = skalski.build_table(per_frame, smooth=True)   # exponential track smoothing
schema.validate(df)

# --- inline sanity asserts (FEET trap + presence + stable tracks) ---
assert df[~df.is_ball].x.between(0, 94).all() and df[~df.is_ball].y.between(0, 50).all(), "FEET trap: coords out of court"
assert (df.groupby("frame").size() >= 2).mean() > 0.9, "too few rows/frame"
assert df.is_ball.mean() > 0.1, "ball missing >90% of frames"
stable = (df[~df.is_ball].groupby("track_id").frame.count() > len(df.frame.unique()) * 0.5).sum()
assert stable >= 2, "no stable tracks (SAM2 ids not wired?)"

print("rows:", len(df), "| frames:", df.frame.nunique(), "| tracks:", df.track_id.nunique())
df.head()


## 11. Brain + render (our value layer)

The matchup engine reads who guards whom and the Beaten Index, then renders the X-ray GIF — our layer on top of the Skalski coordinates.

In [ ]:
results, off_team, hoop = brain.run(df)
render.render_gif(results, df, "/content/matchup_engine.gif")

from IPython.display import display, Image
display(Image("/content/matchup_engine.gif"))


## 12. Shot events — make/miss (Skalski make-or-miss notebook, cell 58)

`ShotEventTracker` watches the jump-shot / layup-dunk / ball-in-basket classes and emits
SHOT_STARTED / MADE / MISSED events with frame indices. Verbatim params (fps-scaled).

In [ ]:
video_info = sv.VideoInfo.from_video_path(VIDEO)
shot_event_tracker = ShotEventTracker(
    reset_time_frames=int(video_info.fps * 1.7),
    minimum_frames_between_starts=int(video_info.fps * 0.5),
    cooldown_frames_after_made=int(video_info.fps * 0.5),
)

shot_events = []
for frame_index, frame in enumerate(tqdm(sv.get_video_frames_generator(VIDEO), desc="shots")):
    result = PLAYER_DETECTION_MODEL.infer(
        frame, confidence=PLAYER_DETECTION_MODEL_CONFIDENCE,
        iou_threshold=PLAYER_DETECTION_MODEL_IOU_THRESHOLD)[0]
    detections = sv.Detections.from_inference(result)

    events = shot_event_tracker.update(
        frame_index=frame_index,
        has_jump_shot=len(detections[detections.class_id == JUMP_SHOT_CLASS_ID]) > 0,
        has_layup_dunk=len(detections[detections.class_id == LAYUP_DUNK_CLASS_ID]) > 0,
        has_ball_in_basket=len(detections[detections.class_id == BALL_IN_BASKET_CLASS_ID]) > 0,
    )
    if events:
        shot_events.append((frame_index, events))
        print(frame_index, events)

print("shot events:", shot_events)


## 13. Save artifacts

Pickle `frames_out` (re-run the brain locally with `run_video.py` / `validate_video.py`) and download whatever GIFs were produced.

In [ ]:
import pickle
with open("/content/frames_out.pkl", "wb") as fh:
    pickle.dump(frames_out, fh)

from google.colab import files
for path in [
    "/content/frames_out.pkl",
    "/content/skalski_footage_compressed.mp4",
    "/content/matchup_engine.gif",
]:
    if os.path.exists(path):
        files.download(path)


## Next

- T4 low-VRAM: SAM2 `tiny` (cell 6) + `RUN_OCR=False` (cell 9).
- Save the headline GIF to `prototypes/video_matchup.gif` and stills to `read_the_floor/frames/`
  locally via `run_video.py --frames-out frames_out.pkl`.
- Stack the Value Surface (EPV) + Risk on top of the same tracking table.
